# 01探索性数据分析

该note用于检查恒星类型预测项目的原始数据，包括数据规模、字段结构、目标变量、缺失值、重复值、变量类型和初步分布情况。

In [ ]:
#导入库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [ ]:
# 定义数据路径
DATA_DIR = Path("../data/raw")

train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
submission_path = DATA_DIR / "sample_submission.csv"

print(train_path)
print(test_path)
print(submission_path)

In [ ]:
# 加载数据
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(submission_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

In [ ]:
# 显示训练数据的前几行
train.head()
# 显示测试数据的前几行
test.head()
# 显示提交文件的前几行
sample_submission.head()

In [ ]:
# 显示列名
print("Train columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nSample submission columns:")
print(sample_submission.columns.tolist())

列名	                    含义
id	              每条样本的唯一编号，只用于对应结果，通常不作为模型特征
alpha	          赤经，天体在天空中的横向坐标，类似经度
delta	          赤纬，天体在天空中的纵向坐标，类似纬度
u, g, r, i, z	  五个光学波段/滤镜下的亮度或星等特征，可反映颜色和光谱差异
redshift	      红移，表示光谱波长被拉长的程度，常和天体远近、退行速度有关
spectral_type	  光谱类型，分类特征
galaxy_population 星系/天体族群类别，分类特征
class	          目标标签，也就是模型要预测的类别

In [ ]:
# 找出训练数据中存在但测试数据中不存在的列，这些列可能是目标变量
possible_target_cols = [col for col in train.columns if col not in test.columns]

print("Possible target columns:")
print(possible_target_cols)

In [ ]:
# 显示训练数据的基本信息
train.info()
# 显示测试数据的基本信息
test.info()

In [ ]:
# 显示训练数据的统计信息
train.describe()

In [ ]:
# 检查缺失值
missing_train = train.isnull().sum().sort_values(ascending=False)
missing_test = test.isnull().sum().sort_values(ascending=False)

print("Missing values in train:")
print(missing_train[missing_train > 0])

print("\nMissing values in test:")
print(missing_test[missing_test > 0])

In [ ]:
# 检查重复值
print("Duplicated rows in train:", train.duplicated().sum())
print("Duplicated rows in test:", test.duplicated().sum())

In [ ]:
# 手动设置目标列
target_col = possible_target_cols[0]
print("Target column:", target_col)
# 显示目标列的分布
train[target_col].value_counts()
# 看比例
train[target_col].value_counts(normalize=True)

也就是说训练集不是均衡分布，GALAXY 占了接近三分之二，STAR 只有约 14%。模型如果只追求 accuracy，可能会偏向预测成 GALAXY。

In [ ]:
#画目标变量分布图
train[target_col].value_counts().sort_index().plot(kind="bar")

plt.title("Target Distribution")
plt.xlabel(target_col)
plt.ylabel("Count")
plt.show()

In [ ]:
#区分数值变量和类别变量
id_cols = ["id"] if "id" in train.columns else []

feature_cols = [col for col in train.columns if col not in id_cols + [target_col]]

num_cols = train[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = train[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()

print("ID columns:", id_cols)
print("Feature columns:", feature_cols)
print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

数值变量
['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
alpha、delta：天体坐标
u, g, r, i, z：不同波段的亮度/星等
redshift：红移

分类变量
['spectral_type', 'galaxy_population']
spectral_type	  光谱类型，分类特征
galaxy_population 星系/天体族群类别，分类特征


In [ ]:
#数值变量分布图
from pathlib import Path

fig_dir = Path("../reports/eda/figures/numeric_distributions")
fig_dir.mkdir(parents=True, exist_ok=True)

for col in num_cols:
    train[col].hist(bins=30)

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")

    plt.savefig(fig_dir / f"distribution_{col}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
# 可视化类别变量的分布
for col in cat_cols:
    print(f"\n{col}")
    print(train[col].value_counts())

In [ ]:
# 特征和目标变量的关系
from pathlib import Path

# reports目录在项目根目录下
REPORT_DIR = Path("../reports/eda")
TABLE_DIR = REPORT_DIR / "tables"
FIGURE_DIR = REPORT_DIR / "figures"

# 自动创建文件夹
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("TABLE_DIR:", TABLE_DIR.resolve())
print("FIGURE_DIR:", FIGURE_DIR.resolve())
excel_path = TABLE_DIR / "numeric_features_by_target_describe.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for col in num_cols:
        summary = train.groupby(target_col)[col].describe()
        
        display(summary)
        
        # 保存单独CSV
        csv_path = TABLE_DIR / f"{col}_by_{target_col}_describe.csv"
        summary.to_csv(csv_path, encoding="utf-8-sig")
        
        # 同时写入Excel不同sheet
        summary.to_excel(writer, sheet_name=col[:31])

print(f"All summary tables saved to: {excel_path}")

这些表格是 数值特征按目标类别 class 分组后的统计描述。

比如：

alpha_by_class_describe.csv
表示：分别统计 GALAXY、QSO、STAR 三类样本中，alpha 这个数值变量的分布情况。

每个 CSV 对应一个数值变量：

alpha_by_class_describe.csv      赤经 alpha
delta_by_class_describe.csv      赤纬 delta
u_by_class_describe.csv          u 波段亮度/星等
g_by_class_describe.csv          g 波段亮度/星等
r_by_class_describe.csv          r 波段亮度/星等
i_by_class_describe.csv          i 波段亮度/星等
z_by_class_describe.csv          z 波段亮度/星等
redshift_by_class_describe.csv   红移

<div style="font-size:16px; line-height:1.7;">

## EDA阶段性结论

通过对训练集目标变量和主要特征变量的探索性分析，可以得到以下阶段性结论。

首先，训练集存在明显的类别不均衡问题。其中，GALAXY样本占比最高，约为65.38%；QSO样本约占20.29%；STAR样本约占14.33%。这说明如果后续模型仅以accuracy作为评价指标，模型可能倾向于预测多数类GALAXY，从而掩盖其对少数类STAR和QSO的识别不足。因此，在后续建模阶段，除accuracy外，还需要重点关注balanced accuracy、macro F1-score、classification report和confusion matrix等指标，以更全面地评价模型在各类别上的分类能力。

其次，从数值变量的分组描述统计结果看，redshift在三类目标之间表现出较强的区分能力。STAR的redshift整体接近0，GALAXY处于中等水平，而QSO的redshift明显更高，三类之间呈现较清晰的梯度差异。因此，redshift很可能是后续模型中最重要的判别特征之一。不过，redshift在不同类别之间仍存在一定重叠，并且部分样本存在较大的极端值，因此不能简单地用单一阈值完成分类，仍需要结合其他变量进行综合判断。

再次，u、g、r、i、z等不同波段的亮度/星等变量在不同类别之间也存在明显差异。STAR在部分波段上的数值整体较低，QSO在r、i、z等波段上的数值整体偏高，而GALAXY则介于两者之间或呈现不同的分布特征。这说明光度变量对恒星类型预测具有较强的信息含量。由于天体分类往往与不同波段之间的相对差异有关，因此后续可以进一步构造颜色指数特征，例如u_g、g_r、r_i和i_z，以增强模型对光谱差异的刻画能力。

此外，alpha和delta作为天体坐标变量，在三类之间的均值虽然存在一定差别，但类内波动较大，分布重叠明显，单变量区分能力相对有限。从物理解释角度看，它们不是最核心的分类特征；但从机器学习建模角度看，空间坐标变量可能隐含观测区域、采样机制或数据分布差异，因此暂时不应直接删除，而应在后续模型训练和特征重要性分析中进一步判断其实际贡献。

最后，spectral_type和galaxy_population作为分类特征，也应在后续建模中保留并进行适当编码。这类变量可能包含与天体物理属性相关的类别信息，对提升模型分类性能具有潜在价值。综合当前EDA结果，后续Baseline模型应优先纳入redshift、u、g、r、i、z、spectral_type、galaxy_population等原始特征，并尝试构造颜色指数特征；同时在训练验证划分时采用stratify分层抽样，在模型评价时避免只依赖accuracy。

</div>